# Extended Kalman filter: notebook

The corridor robot of the Kalman-filter Note, now located by its straight-line distance to a beacon on the wall at 6 m, 2 m to the side (reading noise sd 0.1 m). We check every number in the Note and run the experiments behind its figures.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import chi2

import numpy as np

B, D = 6.0, 2.0                 # beacon position along the corridor, and its distance to the side (m)
U, Q, R = 0.5, 0.01, 0.01
X0, P0 = 0.0, 1.0
X_TRUE = np.array([0.73, 1.4, 1.74, 2.17, 2.51, 3.04, 3.57, 3.92, 4.23, 4.7, 5.13, 5.67, 6.32, 6.82, 7.39, 7.86])
Z = np.array([5.61, 5.08, 4.71, 4.33, 4.04, 3.73, 3.2, 3.11, 2.78, 2.3, 2.11, 2.02, 1.84, 2.07, 2.23, 2.75])


def h(x):
    """Measurement model: distance from the robot at x to the beacon (m)."""
    return np.sqrt((B - x) ** 2 + D ** 2)


def dh(x):
    """Its slope (derivative): how much the distance changes per metre driven."""
    return -(B - x) / h(x)


def ekf(z=Z, q=Q, r=R, x=X0, p=P0, u=U):
    """The one-dimensional extended Kalman filter. One dict per step."""
    rows = []
    for zk in z:
        xb, pb = x + u, p + q                    # predict (the motion is linear here)
        H = dh(xb)                               # slope of h at the prediction
        S = H * pb * H + r                       # innovation variance
        K = pb * H / S                           # gain
        y = zk - h(xb)                           # innovation
        x, p = xb + K * y, (1 - K * H) * pb
        rows.append(dict(x_pred=xb, p_pred=pb, H=H, h_pred=h(xb), S=S, K=K, y=y, x=x, p=p))
    return rows


def simulate(n, T=16, x0_sd=1.0):
    """A fresh run: true start drawn around the first guess, slips and sensor noise drawn with seed n."""
    rg = np.random.default_rng(1000 + n)
    xt, xs, zs = rg.normal(X0, x0_sd), [], []
    for _ in range(T):
        xt += U + rg.normal(0, np.sqrt(Q))
        xs.append(xt)
        zs.append(h(xt) + rg.normal(0, np.sqrt(R)))
    return np.array(xs), np.array(zs)


def consistency(q, r, N=500):
    """Average NEES and NIS per step over N simulated runs, for a filter that assumes noises q and r."""
    nees, nis = [], []
    for n in range(N):
        xs, zs = simulate(n)
        rows = ekf(zs, q=q, r=r)
        nees.append([(t - row["x"]) ** 2 / row["p"] for t, row in zip(xs, rows)])
        nis.append([row["y"] ** 2 / row["S"] for row in rows])
    return np.mean(nees, 0), np.mean(nis, 0)

## 1. A bell pushed through the curve (Note, Section 3)

200,000 samples of the position pushed through h, against the tangent-line answer.

In [2]:
rng = np.random.default_rng(0)
rows = []
for mu, s in [(1.0, 0.5), (5.5, 0.5), (5.5, 1.5)]:
    y = h(rng.normal(mu, s, 200_000))
    rows.append(dict(mean_in=mu, sd_in=s, true_mean=y.mean().round(3), true_sd=y.std().round(3),
                     tangent_mean=round(h(mu), 3), tangent_sd=round(abs(dh(mu)) * s, 3),
                     skewness=round(((y - y.mean())**3).mean() / y.std()**3, 2)))
pd.DataFrame(rows)

,mean_in,sd_in,true_mean,true_sd,tangent_mean,tangent_sd,skewness
0,1.0,0.5,5.388,0.464,5.385,0.464,0.05
1,5.5,0.5,2.117,0.137,2.062,0.121,1.89
2,5.5,1.5,2.483,0.574,2.062,0.364,1.88


## 2. A pose through the motion model (Note, Section 3.4)

In [3]:
th, d = np.radians(30), 1.0
G = np.array([[1, 0, -d * np.sin(th)], [0, 1, d * np.cos(th)], [0, 0, 1]])
P = np.diag([0.01, 0.01, 0.09])
print("G =\n", G.round(3)); print("G P G^T =\n", (G @ P @ G.T).round(4))
N = 200_000
X, Y, T = rng.normal(2, 0.1, N), rng.normal(1, 0.1, N), rng.normal(th, 0.3, N)
X2, Y2 = X + d * np.cos(T), Y + d * np.sin(T)
print("samples: mean", X2.mean().round(3), Y2.mean().round(3), " covariance\n", np.cov(X2, Y2).round(4))

G =
 [[ 1.     0.    -0.5  ]
 [ 0.     1.     0.866]
 [ 0.     0.     1.   ]]
G P G^T =
 [[ 0.0325 -0.039  -0.045 ]
 [-0.039   0.0775  0.0779]
 [-0.045   0.0779  0.09  ]]
samples: mean 2.828 1.478  covariance
 [[ 0.0333 -0.0339]
 [-0.0339  0.0723]]


## 3. Sixteen steps on the corridor (Note, Section 4)

In [4]:
df = pd.DataFrame(ekf()).round(4)
df.insert(0, "step", range(1, 17)); df["z"] = Z; df["true"] = X_TRUE
df

,step,x_pred,p_pred,H,h_pred,S,K,y,x,p,z,true
0,1,0.5000,1.0100,-0.9398,5.8523,0.9020,-1.0523,-0.2423,0.7550,0.0112,5.61,0.73
1,2,1.2550,0.0212,-0.9215,5.1493,0.0280,-0.6976,-0.0693,1.3033,0.0076,5.08,1.40
2,3,1.8033,0.0176,-0.9027,4.6489,0.0243,-0.6522,0.0611,1.7635,0.0072,4.71,1.74
3,4,2.2635,0.0172,-0.8816,4.2381,0.0234,-0.6493,0.0919,2.2038,0.0074,4.33,2.17
4,5,2.7038,0.0174,-0.8549,3.8555,0.0227,-0.6542,0.1845,2.5831,0.0077,4.04,2.51
5,6,3.0831,0.0177,-0.8247,3.5367,0.0220,-0.6615,0.1933,2.9552,0.0080,3.73,3.04
6,7,3.4552,0.0180,-0.7862,3.2366,0.0211,-0.6702,-0.0366,3.4798,0.0085,3.20,3.57
7,8,3.9798,0.0185,-0.7107,2.8428,0.0194,-0.6801,0.2672,3.7980,0.0096,3.11,3.92
8,9,4.2980,0.0196,-0.6481,2.6262,0.0182,-0.6961,0.1538,4.1909,0.0107,2.78,4.23
9,10,4.6909,0.0207,-0.5477,2.3903,0.0162,-0.7003,-0.0903,4.7542,0.0128,2.30,4.70


In [5]:
print("started at 11 m:", np.round([r["x"] for r in ekf(x=11.0)], 2))

started at 11 m: [11.24 11.05 10.78 10.44 10.12  9.79  9.3   9.04  8.7   8.23  7.88  7.65
  7.4   7.42  7.54  7.96]


## 4. Filter honesty (Note, Section 5)

Average NEES and NIS per step over 500 simulated runs, and the 95 percent band for an average of 500 chi-square(1) values.

In [6]:
print("band:", (chi2.ppf([0.025, 0.975], 500) / 500).round(3), "  one value, 95 percent bound:", chi2.ppf(0.95, 1).round(2))
for q, r in [(0.01, 0.01), (0.01, 0.0025), (0.0001, 0.01)]:
    nees, nis = consistency(q, r)
    print(f"Q {q}, R {r}:  NEES mean {nees.mean():.2f} (last {nees[-1]:.2f})   NIS mean {nis.mean():.2f}")

band: [0.88  1.128]   one value, 95 percent bound: 3.84
Q 0.01, R 0.01:  NEES mean 1.00 (last 0.97)   NIS mean 1.01
Q 0.01, R 0.0025:  NEES mean 3.25 (last 3.19)   NIS mean 2.41


Q 0.0001, R 0.01:  NEES mean 15.95 (last 32.94)   NIS mean 2.12


In [7]:
xs, zs = simulate(0, T=60)
for r in (0.01, 0.0025):
    rows = ekf(zs, r=r)
    y = np.array([q["y"] for q in rows]); S = np.array([q["S"] for q in rows])
    print(f"R {r}: share of innovations outside 2 sqrt(S): {np.mean(np.abs(y) > 2 * np.sqrt(S)):.3f}")

R 0.01: share of innovations outside 2 sqrt(S): 0.017
R 0.0025: share of innovations outside 2 sqrt(S): 0.133
